# 01 Check Eelbrain-Main Pipeline Setup

This notebook checks the Eelbrain-main pipeline inputs before estimating any TRFs. It is meant to be run cell-by-cell.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

def find_pipeline_dir(start=Path.cwd()):
    start = Path(start).resolve()
    candidates = [start, *start.parents, start / 'analysis' / 'trf_pipeline']
    for path in candidates:
        if (path / 'alice_eelbrain_main_experiment.py').exists():
            return path
    raise FileNotFoundError(f'Could not find alice_eelbrain_main_experiment.py from {start}')


PIPELINE_DIR = find_pipeline_dir()
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from alice_eelbrain_main_experiment import (
    BIDS_ROOT,
    BIDS_SEGMENT_DURATION,
    PREDICTOR_ROOT,
    SEGMENT_DURATION,
    WAV_SEGMENT_DURATION,
    alice,
)

print(f'Pipeline directory: {PIPELINE_DIR}')
print(f'BIDS root: {BIDS_ROOT}')
print(f'Predictor root: {PREDICTOR_ROOT}')


## Subjects and Segment Durations

`SEGMENT_DURATION` is the duration source used by the TRF pipeline. It now comes from BIDS `events.tsv`. WAV duration is shown only as a QC comparison.

In [ ]:
subjects = alice.get_field_values('subject')
print(f'N subjects: {len(subjects)}')
print(subjects[:10], '...', subjects[-5:])

duration_rows = []
for segment in sorted(SEGMENT_DURATION, key=int):
    bids_duration = BIDS_SEGMENT_DURATION[segment]
    wav_duration = WAV_SEGMENT_DURATION.get(segment)
    duration_rows.append({
        'segment': segment,
        'duration_source_used_by_pipeline': 'BIDS events.tsv',
        'duration_sec': bids_duration,
        'wav_duration_sec': wav_duration,
        'bids_minus_wav_sec': bids_duration - wav_duration if wav_duration is not None else None,
    })

duration_table = pd.DataFrame(duration_rows).sort_values('segment', key=lambda s: s.astype(int))
duration_table

## Predictor Files

The first formal model uses `gammatone-8` files in BIDS derivatives.

In [ ]:
predictor_dir = PREDICTOR_ROOT
rows = []
for segment in range(1, 13):
    path = predictor_dir / f'{segment}~gammatone-8.pickle'
    rows.append({'segment': segment, 'path': str(path), 'exists': path.exists()})

predictor_table = pd.DataFrame(rows)
display(predictor_table)
assert predictor_table['exists'].all(), 'Missing gammatone-8 predictor files'


## Events for One Subject

Eelbrain main reads event timing from BIDS/raw data. The pipeline maps the BIDS `trial_type` marker labels to the clean `segment` variable; `stimulus_id` is printed as a QC cross-check.


In [ ]:
subject = '01'
events = alice.load_events(subject=subject, raw='0.5-20', epoch='story-segments')
print(f'Loaded {events.n_cases} events for subject {subject}')
events.head()


In [ ]:
print('MNE event values:', list(events['value']))
print('BIDS trial types:', list(events['trial_type']))
print('BIDS stimulus IDs:', list(events['stimulus_id']))
print('Clean segment labels:', list(events['segment']))


## Channel-Type Check

`AUD` should remain in raw data as a `misc` channel, not as an EEG target.

In [ ]:
alice.set(subject='01', raw='0.5-20')
raw = alice.load_raw(preload=False)
channel_types = raw.get_channel_types()
print(f'N channels total: {len(raw.ch_names)}')
print(f'N EEG channels: {channel_types.count("eeg")}')
print(f'N MISC channels: {channel_types.count("misc")}')
print(f'AUD present: {"AUD" in raw.ch_names}')
if 'AUD' in raw.ch_names:
    print(f'AUD type: {raw.get_channel_types(picks=["AUD"])[0]}')
